# Orchestration suite aggregation quickstart

Synthetic engineering and validation evidence, not final regulatory capital. This notebook shows the public orchestration handoff: component summaries in, Standardised Approach composition, top-of-house suite capital, attribution summary, and fail-closed validation for inconsistent jurisdiction families.

See also: [orchestration README](../README.md), [module README](../../../docs/modules/frtb-orchestration/README.md), [public API](../../../docs/modules/frtb-orchestration/PUBLIC_API.md), and the root [client integration guide](../../../docs/CLIENT_INTEGRATION.md).

## Public flow

```mermaid
flowchart LR
    upstream[Client risk engine] --> summaries[Public component summaries]
    summaries --> sa[compose_standardised_approach_capital]
    sa --> suite[calculate_suite_capital]
    suite --> attrib[build_suite_attribution_report]
    suite --> store[Result-store handoff]
    invalid[Mixed date / currency / jurisdiction] --> reject[OrchestrationInputError]
    summaries --> invalid
```

## Raw inputs your upstream must emit

For this Tier-1 suite path, orchestration does not consume private component internals. Your upstream or component layer must emit:

| Layer | Public shape | Used here |
| --- | --- | --- |
| SBM / DRC / RRAO | `frtb_common.ComponentCapitalSummary` | SA composition |
| IMA | `frtb_orchestration.ImaCapitalSummary` | Top-of-house suite aggregation |
| CVA | `frtb_orchestration.CvaCapitalSummary` | Top-of-house suite aggregation |
| Attribution | `frtb_common.contribution_bundle.ComponentContributionBundle` | Suite attribution report |

Arrow manifest routing is available separately through `CapitalRunManifest` and `run_standardised_approach_from_manifest`; this notebook focuses on the already-calculated summary handoff.

In [ ]:
from __future__ import annotations

from datetime import date

from frtb_common import (
    CalculationScope,
    CalculationScopeLevel,
    ComponentCapitalSummary,
    StandardisedComponent,
)
from frtb_common.attribution import CapitalContribution  # Package helper
from frtb_common.contribution_bundle import ComponentContributionBundle  # Package helper
from frtb_orchestration import (
    ImaCapitalSummary,
    build_suite_attribution_report,
    calculate_suite_capital,
    compose_standardised_approach_capital,
)
from frtb_orchestration._validation import OrchestrationInputError  # Package helper
from frtb_orchestration.cva_summary import CvaCapitalSummary  # Package helper

AS_OF = date(2026, 6, 3)
BASE_CURRENCY = "USD"

In [ ]:
def make_sa_component(
    component: StandardisedComponent,
    profile_id: str,
    total_capital: float,
    calculation_scope: CalculationScope | None = None,
) -> ComponentCapitalSummary:
    return ComponentCapitalSummary(
        component=component,
        package_name=f"frtb-{component.value.lower()}",
        run_id=f"notebook-{component.value.lower()}-run",
        calculation_date=AS_OF,
        base_currency=BASE_CURRENCY,
        profile_id=profile_id,
        total_capital=total_capital,
        profile_hash=f"profile-{profile_id}",
        input_hash=f"input-{component.value.lower()}",
        line_count=3,
        excluded_line_count=0,
        subtotal_count=1,
        citations=("synthetic-notebook-citation",),
        calculation_scope=calculation_scope,
    )

fallback_scope = CalculationScope(
    level=CalculationScopeLevel.DESK,
    desk_id="credit-desk",
)
sbm_summary = make_sa_component(
    StandardisedComponent.SBM,
    "US_NPR_2_0",
    20.0,
    calculation_scope=fallback_scope,
)
drc_summary = make_sa_component(
    StandardisedComponent.DRC,
    "US_NPR_2_0",
    15.0,
    calculation_scope=fallback_scope,
)
rrao_summary = make_sa_component(
    StandardisedComponent.RRAO,
    "US_NPR_2_0",
    10.0,
    calculation_scope=fallback_scope,
)

sa_result = compose_standardised_approach_capital(
    sbm_summary=sbm_summary,
    drc_summary=drc_summary,
    rrao_summary=rrao_summary,
    ima_desk_eligibility={"rates-desk": "IMA_ELIGIBLE", "credit-desk": "SA_FALLBACK"},
    run_id="notebook-sa-run",
)

print(f"SA total: {sa_result.total_capital:.2f} {sa_result.base_currency}")
print(f"Jurisdiction family: {sa_result.jurisdiction_family}")
print(f"Fallback route count: {len(sa_result.fallback_routes)}")

## Suite capital

The top-of-house path is additive: IMA + SA + CVA. Component packages own their own multipliers, floors, method eligibility, and regulatory validation before orchestration receives these summary shapes.

In [ ]:
ima_summary = ImaCapitalSummary(
    package_name="frtb-ima",
    run_id="notebook-ima-run",
    calculation_date=AS_OF,
    base_currency=BASE_CURRENCY,
    profile_id="FED_NPR_2_0",
    total_ima_capital=80.0,
    ima_eligible_desk_count=1,
    sa_fallback_desk_count=1,
    policy_hash="policy-fed-npr",
    input_hash="input-ima-scenario-cube",
    citations=("synthetic-ima-citation",),
)

cva_summary = CvaCapitalSummary(
    package_name="frtb-cva",
    run_id="notebook-cva-run",
    calculation_date=AS_OF,
    base_currency=BASE_CURRENCY,
    profile_id="US_NPR20_VB",
    method="BA_CVA_REDUCED",
    total_cva_capital=17.0,
    ba_cva_reduced_total=17.0,
    ba_cva_full_total=None,
    sa_cva_total=None,
    profile_hash="profile-us-npr20-cva",
    input_hash="input-cva-exposures",
    risk_class_count=0,
    counterparty_count=2,
    citations=("synthetic-cva-citation",),
)

suite_result = calculate_suite_capital(
    ima_summary=ima_summary,
    sa_result=sa_result,
    cva_summary=cva_summary,
    run_id="notebook-suite-run",
)

for label, amount in (
    ("IMA", suite_result.ima_capital),
    ("SA", suite_result.sa_capital),
    ("CVA", suite_result.cva_capital),
    ("Total", suite_result.total_capital),
):
    print(f"{label}: {amount:.2f} {suite_result.base_currency}")

## Attribution handoff

Component contribution bundles can be aggregated without mutating the component records. The suite layer adds an explicit residual record so downstream result-store or reporting tools can reconcile the bundle set to top-of-house capital.

In [ ]:
def make_component_bundle(component: str, total: float) -> ComponentContributionBundle:
    contribution = CapitalContribution(
        contribution_id=f"{component.lower()}-standalone",
        source_id=component.lower(),
        source_level="COMPONENT",
        bucket_key=None,
        category=component,
        base_amount=total,
        marginal_multiplier=None,
        contribution=total,
        method="STANDALONE",
        citations=("ADR 0038",),
        reconciliation_status="RECONCILED",
    )
    return ComponentContributionBundle(
        component=component,
        contributions=(contribution,),
        component_total=total,
        component_input_hash=f"input-{component.lower()}",
        component_profile_hash=f"profile-{component.lower()}",
    )

report = build_suite_attribution_report(
    suite_result=suite_result,
    component_bundles=(
        make_component_bundle("IMA", suite_result.ima_capital),
        make_component_bundle("SA", suite_result.sa_capital),
        make_component_bundle("CVA", suite_result.cva_capital),
    ),
)

print(f"Attribution component set: {', '.join(report.component_set)}")
print(f"Contribution records including residual: {len(report.contribution_records)}")
print(f"Residual status: {report.reconciliation_status.value}")

## Fail-closed validation

The suite layer rejects mixed jurisdiction families instead of silently aggregating incomparable results.

In [ ]:
eu_cva_summary = CvaCapitalSummary(
    package_name="frtb-cva",
    run_id="notebook-cva-eu-run",
    calculation_date=AS_OF,
    base_currency=BASE_CURRENCY,
    profile_id="EU_CRR3_CVA",
    method="BA_CVA_REDUCED",
    total_cva_capital=17.0,
    ba_cva_reduced_total=17.0,
    ba_cva_full_total=None,
    sa_cva_total=None,
    profile_hash="profile-eu-crr3-cva",
    input_hash="input-cva-exposures",
    risk_class_count=0,
    counterparty_count=2,
    citations=("synthetic-eu-cva-citation",),
)

try:
    calculate_suite_capital(
        ima_summary=ima_summary,
        sa_result=sa_result,
        cva_summary=eu_cva_summary,
    )
except OrchestrationInputError as exc:
    print(f"Rejected as expected: {exc}")